# MLB data processing: 2023–2025 play-by-play + team data

Builds a one-row-per-game modeling table from Retrosheet play-by-play data and team-level data.

**Steps**
1. Load the play-by-play and team files and keep regular-season games only
2. Build game-level fields: final score, home win, stats through 6 innings
3. Season-to-date win % and performance index (games *before* each game, same season only)
4. Add the new fields to the play-by-play table
5. Remove games that ended in the 8th or earlier, went to extra innings, or had a run differential outside −4 to +4 after 6
6. Build the one-row-per-game table
7. Merge innings 7–9 from the team data and add season-to-date late runs per game

**Conventions**
- "Up to that point" means games played *before* the current game in the same season. The current game is never included.
- A team's first game of a season has no prior games, so its win % and late runs are set to 0.
- Team game order within a season is by date, then doubleheader game number (last digit of `gid`).
- Season-to-date values are computed from **all** regular-season games, before the inning and run-differential filters, so filtered-out games still count toward each team's record.

## Setup
Set the file paths. Each path setting takes one file or a list of files (for example, one file per season).

In [31]:
import numpy as np
import pandas as pd

# Play-by-play file(s) for 2023, 2024 and 2025 (one combined file or one per season)
PBP_PATHS = ["23_24_25plays.csv"]
# Team data file(s) for the same seasons (Retrosheet teamstats format)
TEAM_PATHS = ["23_24_25teamstats.csv"]
# Where to save the finished one-row-per-game table
OUT_PATH = "23_24_25games_processed.csv"

THROUGH_INNING = 6          # stats are measured through this inning
MAX_ABS_RUN_DIFF = 4        # keep games with -4 <= run differential <= 4
REGULATION_INNINGS = 9

HIT_COLS = ["single", "double", "triple", "hr"]
STEAL_COLS = ["sb2", "sb3", "sbh"]
ERROR_COLS = [f"e{i}" for i in range(1, 10)]


def as_list(paths):
    return [paths] if isinstance(paths, str) else list(paths)


def read_many(paths, **kwargs):
    """Read one or more CSV files and stack them."""
    return pd.concat([pd.read_csv(p, **kwargs) for p in as_list(paths)], ignore_index=True)

## 1. Load the play-by-play data and keep regular-season games

In [32]:
plays = read_many(PBP_PATHS, dtype={"gid": "string", "batteam": "string", "fldteam": "string",
                                    "gametype": "string", "event": "string"},
                  low_memory=False)
print(f"Loaded {len(plays):,} plays, {plays['gid'].nunique():,} games, {plays.shape[1]} columns")

plays = plays.copy()   # consolidate memory after reading
plays["date"] = pd.to_datetime(plays["date"].astype(str), format="%Y%m%d")
plays = plays.assign(season=plays["date"].dt.year,
                     game_num=plays["gid"].str[-1].astype(int))   # 0 = single game, 1/2 = doubleheader

print("\nPlays by game type:")
print(plays["gametype"].value_counts())

plays = plays[plays["gametype"] == "regular"].copy()
plays = plays.sort_values(["date", "gid", "pn"], kind="stable").reset_index(drop=True)
print(f"\nAfter keeping regular season only: {len(plays):,} plays, {plays['gid'].nunique():,} games")
print(plays.groupby("season")["gid"].nunique().rename("games per season"))

Loaded 581,115 plays, 7,423 games, 179 columns

Plays by game type:
gametype
regular           570548
divisionseries      3907
lcs                 2877
wildcard            2103
worldseries         1451
allstar              229
Name: count, dtype: Int64

After keeping regular season only: 570,548 plays, 7,289 games
season
2023    2430
2024    2429
2025    2430
Name: games per season, dtype: int64


### Home and visiting team codes

`home_team` and `away_team` are derived for every play from the batting and fielding team columns. When the visitors bat (`vis_home` = 0), the home team is `fldteam` and the visitors are `batteam`; when the home team bats, it's the reverse. Each game must have exactly one home team and one visiting team.

In [33]:
plays["home_team"] = np.where(plays["vis_home"] == 0, plays["fldteam"], plays["batteam"])
plays["away_team"] = np.where(plays["vis_home"] == 0, plays["batteam"], plays["fldteam"])

teams_per_game = plays.groupby("gid")[["home_team", "away_team"]].nunique()
assert (teams_per_game == 1).all().all(), "A game has more than one home or visiting team code"
assert (plays["home_team"] != plays["away_team"]).all()
print("Home team matches the gid prefix in "
      f"{(plays['gid'].str[:3] == plays['home_team']).mean():.2%} of plays")
plays[["gid", "inning", "vis_home", "batteam", "fldteam", "home_team", "away_team"]].head()

Home team matches the gid prefix in 100.00% of plays


,gid,inning,vis_home,batteam,fldteam,home_team,away_team
0,BOS202303300,1,0,BAL,BOS,BOS,BAL
1,BOS202303300,1,0,BAL,BOS,BOS,BAL
2,BOS202303300,1,0,BAL,BOS,BOS,BAL
3,BOS202303300,1,0,BAL,BOS,BOS,BAL
4,BOS202303300,1,0,BAL,BOS,BOS,BAL


## 2. Game-level fields

For every regular-season game:
- **Final score:** the score before the last play plus the runs scored on that play.
- **Stats through the 6th inning:** batting stats (hits, walks, steals, runs) count plays where that team is batting. Fielding stats (strikeouts, errors) count plays where that team is fielding, i.e. the other team is batting. `vis_home` is 0 when the visitors bat and 1 when the home team bats.

In [34]:
# Teams, date and season per game
games_base = plays.groupby("gid").agg(home_team=("home_team", "first"), away_team=("away_team", "first"),
                                      date=("date", "first"), season=("season", "first"),
                                      game_num=("game_num", "first"), max_inning=("inning", "max"))

# Final score = score before the last play + runs on that play
last = plays.groupby("gid").tail(1).set_index("gid")
games_base["home_final_score"] = last["score_h"] + np.where(last["vis_home"] == 1, last["runs"], 0)
games_base["away_final_score"] = last["score_v"] + np.where(last["vis_home"] == 0, last["runs"], 0)
games_base["home_win"] = (games_base["home_final_score"] > games_base["away_final_score"]).astype(int)

ties = (games_base["home_final_score"] == games_base["away_final_score"]).sum()
print(f"{len(games_base):,} games; {ties} ended tied (counted as home_win = 0)")
games_base.head()

7,289 games; 0 ended tied (counted as home_win = 0)


,home_team,away_team,date,season,game_num,max_inning,home_final_score,away_final_score,home_win
gid,,,,,,,,,
ANA202304070,ANA,TOR,2023-04-07,2023,0,9,3,4,0
ANA202304080,ANA,TOR,2023-04-08,2023,0,9,9,5,1
ANA202304090,ANA,TOR,2023-04-09,2023,0,10,11,12,0
ANA202304100,ANA,WAS,2023-04-10,2023,0,9,4,6,0
ANA202304110,ANA,WAS,2023-04-11,2023,0,9,2,0,1


In [35]:
# Stats through the 6th inning
p6 = plays[plays["inning"] <= THROUGH_INNING].copy()
p6["hits"] = p6[HIT_COLS].sum(axis=1)
p6["steals"] = p6[STEAL_COLS].sum(axis=1)
p6["errors"] = p6[ERROR_COLS].sum(axis=1)

by_half = (p6.groupby(["gid", "vis_home"])[["runs", "hits", "walk", "steals", "k", "errors"]]
           .sum().unstack("vis_home", fill_value=0))
by_half = by_half.reindex(columns=pd.MultiIndex.from_product([by_half.columns.levels[0], [0, 1]]),
                          fill_value=0)

through6 = pd.DataFrame(index=by_half.index)
# Batting stats: credited to the team at bat (vis_home 1 = home batting, 0 = visitors batting)
through6["home_hits_6"] = by_half[("hits", 1)]
through6["away_hits_6"] = by_half[("hits", 0)]
through6["walks_h_batting_6"] = by_half[("walk", 1)]
through6["walks_a_batting_6"] = by_half[("walk", 0)]
through6["home_steals_6"] = by_half[("steals", 1)]
through6["away_steals_6"] = by_half[("steals", 0)]
# Fielding stats: credited to the team in the field (home fields when visitors bat)
through6["k_h_pitching_6"] = by_half[("k", 0)]
through6["k_a_pitching_6"] = by_half[("k", 1)]
through6["home_errors_6"] = by_half[("errors", 0)]
through6["away_errors_6"] = by_half[("errors", 1)]
# Score through 6 innings, from runs scored on plays in innings 1-6
home_runs_6 = by_half[("runs", 1)]
away_runs_6 = by_half[("runs", 0)]
through6["run_diff_6"] = home_runs_6 - away_runs_6          # home minus visitor
through6["total_runs_6"] = home_runs_6 + away_runs_6

games_base = games_base.join(through6, how="left")
games_base[through6.columns] = games_base[through6.columns].fillna(0).astype(int)
games_base[list(through6.columns)].describe().round(2)

,home_hits_6,away_hits_6,walks_h_batting_6,walks_a_batting_6,home_steals_6,away_steals_6,k_h_pitching_6,k_a_pitching_6,home_errors_6,away_errors_6,run_diff_6,total_runs_6
count,7289.00,7289.00,7289.00,7289.00,7289.00,7289.00,7289.00,7289.00,7289.00,7289.00,7289.00,7289.00
mean,5.81,5.61,2.11,1.97,0.49,0.48,5.73,5.49,0.34,0.36,0.23,6.08
std,2.84,2.84,1.62,1.56,0.82,0.81,2.32,2.29,0.61,0.63,3.72,3.67
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-24.00,0.00
25%,4.00,4.00,1.00,1.00,0.00,0.00,4.00,4.00,0.00,0.00,-2.00,3.00
50%,5.00,5.00,2.00,2.00,0.00,0.00,6.00,5.00,0.00,0.00,0.00,6.00
75%,8.00,7.00,3.00,3.00,1.00,1.00,7.00,7.00,1.00,1.00,2.00,8.00
max,19.00,25.00,10.00,12.00,9.00,8.00,15.00,14.00,5.00,4.00,16.00,26.00


## 3. Season-to-date win % and performance index

Each game is split into two team-game rows (home and visitor), sorted by date and game number within each **season and team**. Wins and games *before* each game are cumulative counts that restart every season, so 2024 values only use 2024 games.

Performance index = home win % / visiting win %. If the visiting win % is 0, it is the home team's number of wins so far that season.

In [36]:
team_games = pd.concat([
    games_base.reset_index()[["gid", "season", "date", "game_num", "home_team", "home_win"]]
        .rename(columns={"home_team": "team", "home_win": "win"}).assign(side="home"),
    games_base.reset_index()[["gid", "season", "date", "game_num", "away_team", "home_win"]]
        .rename(columns={"away_team": "team"}).assign(win=lambda d: 1 - d["home_win"], side="away")
        .drop(columns="home_win"),
], ignore_index=True)

team_games = team_games.sort_values(["season", "team", "date", "game_num"], kind="stable")
grp = team_games.groupby(["season", "team"], sort=False)
team_games["wins_before"] = grp["win"].cumsum() - team_games["win"]     # exclude current game
team_games["games_before"] = grp.cumcount()
team_games["win_pct_before"] = np.where(team_games["games_before"] > 0,
                                        team_games["wins_before"] / team_games["games_before"].clip(lower=1),
                                        0.0)

# Check: every team's first game of each season starts from zero
firsts = team_games.groupby(["season", "team"]).head(1)
assert (firsts["games_before"] == 0).all() and (firsts["wins_before"] == 0).all()

home_rec = (team_games[team_games["side"] == "home"].set_index("gid")
            [["win_pct_before", "wins_before"]]
            .rename(columns={"win_pct_before": "home_win_pct", "wins_before": "home_wins_before"}))
away_rec = (team_games[team_games["side"] == "away"].set_index("gid")
            [["win_pct_before"]].rename(columns={"win_pct_before": "away_win_pct"}))
games_base = games_base.join(home_rec).join(away_rec)

games_base["home_perf_index"] = np.where(
    games_base["away_win_pct"] == 0,
    games_base["home_wins_before"],
    games_base["home_win_pct"] / games_base["away_win_pct"].replace(0, np.nan),
)
games_base[["home_team", "away_team", "home_win_pct", "away_win_pct", "home_perf_index"]].describe().round(3)

,home_win_pct,away_win_pct,home_perf_index
count,7289.000,7289.000,7289.000
mean,0.497,0.497,1.064
std,0.124,0.125,0.501
min,0.000,0.000,0.000
25%,0.450,0.449,0.836
50%,0.505,0.506,1.000
75%,0.563,0.564,1.189
max,1.000,1.000,10.000


## 4. Add the new fields to the play-by-play table

In [37]:
NEW_FIELDS = ["run_diff_6", "home_win_pct", "away_win_pct",
              "home_final_score", "away_final_score", "home_perf_index", "home_win",
              "home_hits_6", "away_hits_6", "walks_h_batting_6", "walks_a_batting_6",
              "k_h_pitching_6", "k_a_pitching_6", "home_steals_6", "away_steals_6",
              "home_errors_6", "away_errors_6", "total_runs_6"]

plays = plays.merge(games_base[NEW_FIELDS + ["max_inning"]],
                    left_on="gid", right_index=True, how="left", validate="many_to_one")
assert plays["home_win"].notna().all(), "Some plays did not match a game"
plays[["gid", "inning", "event"] + NEW_FIELDS].head()

,gid,inning,event,run_diff_6,home_win_pct,away_win_pct,home_final_score,away_final_score,home_perf_index,home_win,...,away_hits_6,walks_h_batting_6,walks_a_batting_6,k_h_pitching_6,k_a_pitching_6,home_steals_6,away_steals_6,home_errors_6,away_errors_6,total_runs_6
0,BOS202303300,1,K,-4,0.0,0.0,9,10,0.0,0,...,9,1,8,7,5,0,4,0,0,12
1,BOS202303300,1,HR/F9D,-4,0.0,0.0,9,10,0.0,0,...,9,1,8,7,5,0,4,0,0,12
2,BOS202303300,1,K,-4,0.0,0.0,9,10,0.0,0,...,9,1,8,7,5,0,4,0,0,12
3,BOS202303300,1,W,-4,0.0,0.0,9,10,0.0,0,...,9,1,8,7,5,0,4,0,0,12
4,BOS202303300,1,W.1-2,-4,0.0,0.0,9,10,0.0,0,...,9,1,8,7,5,0,4,0,0,12


## 5. Filter games

- Remove games that ended in the 8th inning or earlier (`max_inning <= 8`)
- Remove games that went to extra innings (`max_inning > 9`)
- Remove games where the run differential after 6 innings is below −4 or above +4

In [38]:
def n_games(df):
    return df["gid"].nunique()

print(f"Regular-season games:                     {n_games(plays):,}")
plays = plays[plays["max_inning"] > 8]
print(f"After removing games ending by the 8th:   {n_games(plays):,}")
plays = plays[plays["max_inning"] <= REGULATION_INNINGS]
print(f"After removing extra-inning games:        {n_games(plays):,}")
plays = plays[plays["run_diff_6"].between(-MAX_ABS_RUN_DIFF, MAX_ABS_RUN_DIFF)]
print(f"After keeping run diff -4 to +4 after 6:  {n_games(plays):,}")
plays = plays.reset_index(drop=True)

Regular-season games:                     7,289
After removing games ending by the 8th:   7,277
After removing extra-inning games:        6,651
After keeping run diff -4 to +4 after 6:  5,185


## 6. One row per game

In [39]:
GAME_COLS = ["gid", "date", "season", "home_team", "away_team"] + NEW_FIELDS

games = (plays.groupby("gid", sort=False).head(1)[GAME_COLS]
         .sort_values(["date", "gid"]).reset_index(drop=True))
assert games["gid"].is_unique
print(games.shape)
print(games.groupby("season").size().rename("games per season"))
games.head()

(5185, 23)
season
2023    1759
2024    1701
2025    1725
Name: games per season, dtype: int64


,gid,date,season,home_team,away_team,run_diff_6,home_win_pct,away_win_pct,home_final_score,away_final_score,...,away_hits_6,walks_h_batting_6,walks_a_batting_6,k_h_pitching_6,k_a_pitching_6,home_steals_6,away_steals_6,home_errors_6,away_errors_6,total_runs_6
0,BOS202303300,2023-03-30,2023,BOS,BAL,-4,0.0,0.0,9,10,...,9,1,8,7,5,0,4,0,0,12
1,CHN202303300,2023-03-30,2023,CHN,MIL,4,0.0,0.0,4,0,...,3,3,3,8,3,0,0,1,1,4
2,CIN202303300,2023-03-30,2023,CIN,PIT,0,0.0,0.0,4,5,...,6,6,7,9,12,0,1,0,1,8
3,HOU202303300,2023-03-30,2023,HOU,CHA,0,0.0,0.0,2,3,...,6,0,0,5,9,1,0,0,0,0
4,KCA202303300,2023-03-30,2023,KCA,MIN,-2,0.0,0.0,0,2,...,9,3,2,5,9,0,0,0,0,2


## 7. Team data: innings 7–9 and season-to-date late runs

The team file has **two rows per game**, one per team. The play-by-play `gid` always starts with the home team's code. The team file's `gid` may start with either team's code, so it isn't matched directly. Each team row is keyed on **team code + date + game number** instead: the `team` column if the file has one (otherwise the first three letters of its `gid`), plus the last 9 characters of `gid` (YYYYMMDD + game number). The home team's row joins on `home_team`, and the visitor's row joins on `away_team`. This works whichever team's code the team file puts at the start of `gid`.

**Late runs** = (inn7 + inn8 + inn9) summed over the team's earlier games this season ÷ games played before this game. A blank inning (e.g., the home team didn't bat in the 9th) counts as 0 runs.

In [40]:
team = read_many(TEAM_PATHS, dtype={"gid": "string"}, low_memory=False)
print(f"Loaded team data: {team.shape}")

# Keep one row per team per game: Retrosheet's stattype marks alternate stat versions
if "stattype" in team.columns and (team["stattype"] == "value").any():
    team = team[team["stattype"] == "value"]
if "gametype" in team.columns:
    team = team[team["gametype"] == "regular"]

team = team.copy()
team["team_code"] = (team["team"] if "team" in team.columns else team["gid"].str[:3]).astype("string")
team["game_key"] = team["gid"].str[-9:]                       # YYYYMMDD + game number
team["season"] = team["game_key"].str[:4].astype(int)
team["t_date"] = pd.to_datetime(team["game_key"].str[:8], format="%Y%m%d")
team["t_game_num"] = team["game_key"].str[-1].astype(int)
for c in ["inn7", "inn8", "inn9"]:
    team[c] = pd.to_numeric(team[c], errors="coerce").fillna(0)

dupes = team.duplicated(["team_code", "game_key"]).sum()
assert dupes == 0, f"{dupes} duplicate team-game rows in the team data"
print(f"{len(team):,} team-game rows (two per game)")

Loaded team data: (14846, 111)
14,578 team-game rows (two per game)


In [41]:
# Season-to-date late runs per game, before the current game, within each season + team
team = team.sort_values(["season", "team_code", "t_date", "t_game_num"], kind="stable")
team["late_runs_game"] = team[["inn7", "inn8", "inn9"]].sum(axis=1)
tgrp = team.groupby(["season", "team_code"], sort=False)
team["late_runs_before"] = tgrp["late_runs_game"].cumsum() - team["late_runs_game"]
team["games_before"] = tgrp.cumcount()
team["late_runs_avg"] = np.where(team["games_before"] > 0,
                                 team["late_runs_before"] / team["games_before"].clip(lower=1), 0.0)

team_cols = team[["team_code", "game_key", "inn7", "inn8", "inn9", "late_runs_avg"]]

games["game_key"] = games["gid"].str[-9:]
before = len(games)
games = (games
    .merge(team_cols.rename(columns={"team_code": "home_team", "inn7": "home_inn7", "inn8": "home_inn8",
                                     "inn9": "home_inn9", "late_runs_avg": "home_late_runs_avg"}),
           on=["home_team", "game_key"], how="left", validate="one_to_one")
    .merge(team_cols.rename(columns={"team_code": "away_team", "inn7": "away_inn7", "inn8": "away_inn8",
                                     "inn9": "away_inn9", "late_runs_avg": "away_late_runs_avg"}),
           on=["away_team", "game_key"], how="left", validate="one_to_one")
    .drop(columns="game_key"))
assert len(games) == before

unmatched = games[["home_inn7", "away_inn7"]].isna().any(axis=1).sum()
print(f"Games without a matching team row: {unmatched}")

games.tail()

Games without a matching team row: 0


,gid,date,season,home_team,away_team,run_diff_6,home_win_pct,away_win_pct,home_final_score,away_final_score,...,away_errors_6,total_runs_6,home_inn7,home_inn8,home_inn9,home_late_runs_avg,away_inn7,away_inn8,away_inn9,away_late_runs_avg
5180,MIL202509280,2025-09-28,2025,MIL,CIN,2,0.596273,0.515528,4,2,...,0,6,0.0,0.0,0.0,1.391304,0.0,0.0,0.0,1.341615
5181,NYA202509280,2025-09-28,2025,NYA,BAL,0,0.577640,0.465839,3,2,...,0,4,0.0,1.0,0.0,1.453416,0.0,0.0,0.0,1.285714
5182,SEA202509280,2025-09-28,2025,SEA,LAN,-4,0.559006,0.571429,1,6,...,0,4,0.0,1.0,0.0,1.472050,1.0,1.0,0.0,1.472050
5183,SFN202509280,2025-09-28,2025,SFN,COL,2,0.496894,0.267081,4,0,...,0,2,0.0,2.0,0.0,1.086957,0.0,0.0,0.0,1.310559
5184,TOR202509280,2025-09-28,2025,TOR,TBA,4,0.577640,0.478261,13,4,...,0,12,5.0,0.0,0.0,1.602484,0.0,0.0,0.0,1.149068


## 8. Final checks and save

In [43]:
FINAL_COLS = (["gid", "date", "season", "home_team", "away_team"] + NEW_FIELDS
              + ["home_inn7", "home_inn8", "home_inn9", "away_inn7", "away_inn8", "away_inn9",
                 "home_late_runs_avg", "away_late_runs_avg"])
games = games[FINAL_COLS]

assert games["gid"].is_unique
assert games["run_diff_6"].between(-MAX_ABS_RUN_DIFF, MAX_ABS_RUN_DIFF).all()
assert (games["gid"].str[:3] == games["home_team"]).mean() > 0.99, "gid prefix should be the home team"

print(games.shape)
print("\nMissing values per column (only columns with any):")
na = games.isna().sum()
print(na[na > 0] if na.any() else "none")
print(f"\nHome team won {games['home_win'].mean():.1%} of {len(games):,} games")

games.to_csv(OUT_PATH, index=False)
print(f"Saved {OUT_PATH}")
games.describe().T.round(3)

(5185, 31)

Missing values per column (only columns with any):
none

Home team won 52.6% of 5,185 games
Saved 23_24_25games_processed.csv


,count,mean,min,25%,50%,75%,max,std
date,5185,2024-06-25 21:52:31.475409920,2023-03-30 00:00:00,2023-08-15 00:00:00,2024-06-25 00:00:00,2025-05-10 00:00:00,2025-09-28 00:00:00,NaN
season,5185.0,2023.993443,2023.0,2023.0,2024.0,2025.0,2025.0,0.819771
run_diff_6,5185.0,0.128255,-4.0,-2.0,0.0,2.0,4.0,2.297329
home_win_pct,5185.0,0.495877,0.0,0.449275,0.505155,0.561798,1.0,0.124563
away_win_pct,5185.0,0.496536,0.0,0.448276,0.506024,0.563636,1.0,0.125055
home_final_score,5185.0,4.0054,0.0,2.0,4.0,6.0,22.0,2.610145
away_final_score,5185.0,4.072517,0.0,2.0,4.0,6.0,18.0,2.853642
home_perf_index,5185.0,1.061697,0.0,0.834259,1.0,1.188571,10.0,0.50389
home_win,5185.0,0.525554,0.0,0.0,1.0,1.0,1.0,0.499395
home_hits_6,5185.0,5.482353,0.0,4.0,5.0,7.0,15.0,2.467874
